# E2E Genomic Model Explorer (Notebook)

Runnable companion to [`e2e_explorer.md`](../e2e_explorer.md). Each section runs a focused PyTorch example from the repo root scripts.


## Embeddings: From Nucleotides to Numbers

**What this step does:** Maps DNA nucleotide tokens to dense vectors so neural networks can learn similarity between bases and motifs.

**Input / Output shapes:** Input: integer indices `(seq_len,)`. Output: embedded tensor `(seq_len, embedding_dim)`.

> **⚙️ Compute note**
> | Step | Typical wall-clock (CPU / single GPU) | Memory footprint | Bottleneck |
> |------|---------------------------------------|------------------|------------|
> | Embedding lookup (len 4, dim 16) | ~0.01 ms / ~0.01 ms | ~1 KB | Negligible |


In [ ]:
import torch
import torch.nn as nn

# --- 1. Setup ---
# Vocabulary: 0=A, 1=C, 2=G, 3=T, 4=N (padding/unknown)
vocab_size = 5
# Each nucleotide will be represented by a vector of size 16
embedding_dim = 16

# Create the embedding layer
embedding_layer = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim)

# --- 2. Example DNA Sequence ---
# Let's represent the sequence "ACGT" as integer indices
# This is our model's input
dna_sequence_indices = torch.tensor([0, 1, 2, 3], dtype=torch.long)

# --- 3. Apply Embedding ---
# Pass the indices through the embedding layer
embedded_sequence = embedding_layer(dna_sequence_indices)

# --- 4. Observe the Output ---
# The output is a tensor where each integer index has been replaced
# by a dense vector of size 'embedding_dim'.
# Shape: (sequence_length, embedding_dim)
print("Shape of embedded sequence:", embedded_sequence.shape)
print("Embedded 'A':\n", embedded_sequence[0])


## Convolution: Finding Motifs in Sequences

**What this step does:** 1D convolutions scan the sequence for short motif patterns such as transcription-factor binding sites.

**Input / Output shapes:** Input: `(batch, channels, length)`. Output: activation map `(batch, out_channels, length_out)`.

> **⚙️ Compute note**
> | Step | Typical wall-clock (CPU / single GPU) | Memory footprint | Bottleneck |
> |------|---------------------------------------|------------------|------------|
> | Conv1d motif scan (batch 1, len 100) | ~5 ms / ~0.5 ms | ~20 MB | Memory bandwidth |


In [ ]:
import torch
import torch.nn as nn

# --- 1. Setup (Continuing from Embedding) ---
batch_size = 1
seq_length = 100
embedding_dim = 16 # This is our number of "input channels"

# A random embedded sequence
# Shape: (batch_size, seq_length, embedding_dim)
embedded_sequence = torch.randn(batch_size, seq_length, embedding_dim)

# PyTorch convolutions expect (Batch, Channels, Length), so we permute the dimensions
embedded_sequence = embedded_sequence.permute(0, 2, 1) # Shape becomes (1, 16, 100)

# --- 2. Create the Convolutional Layer ---
# in_channels: Must match the embedding dimension
# out_channels: The number of different motifs we want to learn (e.g., 32 different scanners)
# kernel_size: The width of the scanner (e.g., a motif of length 8)
conv_layer = nn.Conv1d(in_channels=embedding_dim, out_channels=32, kernel_size=8)

# --- 3. Apply Convolution ---
conv_output = conv_layer(embedded_sequence)

# --- 4. Observe the Output ---
# The output shape will be (batch_size, num_motifs, new_length)
# The length is slightly smaller due to the kernel size.
print("Shape after 1D convolution:", conv_output.shape)


## Attention: Focusing on What Matters

**What this step does:** Self-attention lets each position weigh other positions, capturing long-range regulatory dependencies.

**Input / Output shapes:** Input: `(seq_len, batch, embed_dim)`. Output: context vectors plus `(batch, seq, seq)` weights.

> **⚙️ Compute note**
> | Step | Typical wall-clock (CPU / single GPU) | Memory footprint | Bottleneck |
> |------|---------------------------------------|------------------|------------|
> | Multi-head self-attention (len 100) | ~50 ms / ~2 ms | ~40 MB | Attention O(n²) |


In [ ]:
import torch
import torch.nn as nn

# --- 1. Setup ---
batch_size = 1
seq_length = 100
embedding_dim = 16 # The model's internal dimension

# A random embedded sequence from a previous layer
# Shape: (seq_length, batch_size, embedding_dim) - Attention layers prefer this format
input_sequence = torch.randn(seq_length, batch_size, embedding_dim)

# --- 2. Create the Attention Layer ---
# embed_dim: The model's dimension
# num_heads: How many attention mechanisms to run in parallel. Must be a divisor of embed_dim.
attention_layer = nn.MultiheadAttention(embed_dim=embedding_dim, num_heads=4)

# --- 3. Apply Attention ---
# In self-attention, the query, key, and value are all the same input sequence.
# The model attends to different parts of itself.
# We set need_weights=True to get the attention matrix for interpretation.
attn_output, attn_weights = attention_layer(input_sequence, input_sequence, input_sequence,
                                            need_weights=True)

# --- 4. Observe the Output ---
# attn_output has the same shape as the input, but each position's vector
# is now a context-aware representation.
print("Shape of attention output:", attn_output.shape)

# attn_weights show the learned importance scores.
# Shape: (batch_size, seq_length, seq_length)
# attn_weights[0, i, j] is how much position 'i' paid attention to position 'j'.
print("Shape of attention weights:", attn_weights.shape)


## Attention Heatmap Visualization

**What this step does:** Visualizing attention weights reveals which bases the model treats as interacting partners.

**Input / Output shapes:** Input: self-attention on random embeddings. Output: heatmap figure.

> **⚙️ Compute note**
> | Step | Typical wall-clock (CPU / single GPU) | Memory footprint | Bottleneck |
> |------|---------------------------------------|------------------|------------|
> | Heatmap render (100×100) | ~200 ms / ~50 ms | ~5 MB | CPU plotting |


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn

# Minimal self-attention setup so this script runs standalone.
batch_size = 1
seq_length = 100
embedding_dim = 16
input_sequence = torch.randn(seq_length, batch_size, embedding_dim)
attention_layer = nn.MultiheadAttention(embed_dim=embedding_dim, num_heads=4)
_, attn_weights = attention_layer(
    input_sequence, input_sequence, input_sequence, need_weights=True
)

# Visualize attention weights for the first sequence in the batch.
attention_matrix = attn_weights[0].detach().cpu().numpy()

plt.figure(figsize=(10, 8))
sns.heatmap(attention_matrix, cmap="viridis")
plt.title("Attention Heatmap")
plt.xlabel("Key Positions (Attended To)")
plt.ylabel("Query Positions (Attending From)")
plt.show()


## Transformers: Positional Encodings and Encoder Stack

**What this step does:** Positional encodings restore sequence order; stacked encoder layers build contextual representations.

**Input / Output shapes:** Input: `(seq_len, batch, embed_dim)`. Output: encoded tensor of same shape.

> **⚙️ Compute note**
> | Step | Typical wall-clock (CPU / single GPU) | Memory footprint | Bottleneck |
> |------|---------------------------------------|------------------|------------|
> | Transformer encoder (2 layers, len 100) | ~200 ms / ~5 ms | ~500 MB | Attention O(n²) |


In [ ]:
import torch
import torch.nn as nn
import math

# --- Positional Encoding Module ---
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, dropout: float = 0.1, max_len: int = 5000):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, 1, d_model)
        pe[:, 0, 0::2] = torch.sin(position * div_term)
        pe[:, 0, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x shape: (seq_len, batch_size, d_model)
        x = x + self.pe[:x.size(0)]
        return self.dropout(x)

# --- Transformer Example ---
# 1. Setup
seq_length = 100
embedding_dim = 16
num_heads = 4
num_layers = 2 # Stack 2 transformer layers
batch_size = 1

# Assume we have an embedded sequence
# Shape: (seq_length, batch_size, embedding_dim)
embedded_input = torch.randn(seq_length, batch_size, embedding_dim)

# 2. Add Positional Encodings
pos_encoder = PositionalEncoding(d_model=embedding_dim)
positioned_input = pos_encoder(embedded_input)

# 3. Create the Transformer Encoder
encoder_layer = nn.TransformerEncoderLayer(
    d_model=embedding_dim, 
    nhead=num_heads,
    batch_first=False # Our input is (Seq, Batch, Dim)
)
transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

# 4. Apply the Transformer
output = transformer_encoder(positioned_input)

# 5. Observe the Output
print("Shape of Transformer output:", output.shape)


## Pre-training: K-mer Tokenisation and MLM Masking

**What this step does:** Overlapping k-mers tokenise DNA; masked language modeling teaches the model genomic grammar.

**Input / Output shapes:** Input: DNA string. Output: masked token IDs and label tensor with `-100` on unmasked positions.

> **⚙️ Compute note**
> | Step | Typical wall-clock (CPU / single GPU) | Memory footprint | Bottleneck |
> |------|---------------------------------------|------------------|------------|
> | K-mer tokenisation (1 M bp seq) | ~2 s / ~0.3 s | ~50 MB | CPU string ops |


In [ ]:
import torch
import random
from itertools import product

# --- 1. Vocabulary and Tokenizer Setup ---
def build_kmer_vocabulary(k, special_tokens=['[PAD]', '[UNK]', '[CLS]', '[SEP]', '[MASK]']):
    """Creates a vocabulary for k-mers and special tokens."""
    bases = ['A', 'C', 'G', 'T']
    kmers = [''.join(p) for p in product(bases, repeat=k)]
    vocab = {token: i for i, token in enumerate(special_tokens + kmers)}
    return vocab

def seq_to_kmer_ids(seq, k, vocab):
    """Converts a DNA sequence into a list of k-mer token IDs."""
    kmers = [seq[i:i+k] for i in range(len(seq) - k + 1)]
    # In a real application, you'd add [CLS] and [SEP] here
    # tokenized_kmers = ['[CLS]'] + kmers + ['[SEP]']
    return [vocab.get(token, vocab['[UNK]']) for token in kmers]

# --- 2. Masking Implementation ---
def mask_tokens(token_ids, vocab, mask_prob=0.15):
    """
    Prepares masked inputs and labels for MLM: 80% MASK, 10% random, 10% original.
    Labels are set to -100 for non-masked tokens to be ignored by the loss function.
    """
    inputs = torch.tensor(token_ids, dtype=torch.long)
    labels = inputs.clone()

    # Probability matrix for selecting tokens to mask
    prob_matrix = torch.full(labels.shape, mask_prob)
    
    # Determine which tokens to mask (ignoring special tokens if they were present)
    masked_indices = torch.bernoulli(prob_matrix).bool()
    labels[~masked_indices] = -100  # We only compute loss on masked tokens

    # 80% of masked tokens are replaced with [MASK]
    indices_replaced = torch.bernoulli(torch.full(labels.shape, 0.8)).bool() & masked_indices
    inputs[indices_replaced] = vocab['[MASK]']

    # 10% of masked tokens are replaced with a random token
    # We take 50% of the remaining 20% of masked tokens
    indices_random = torch.bernoulli(torch.full(labels.shape, 0.5)).bool() & masked_indices & ~indices_replaced
    
    # Generate random words, excluding special tokens
    num_special_tokens = 5 # Assuming the 5 standard special tokens
    random_words = torch.randint(num_special_tokens, len(vocab), labels.shape, dtype=torch.long)
    inputs[indices_random] = random_words[indices_random]

    # The remaining 10% are left unchanged implicitly.
    return inputs, labels

# --- 3. Example Usage ---
K = 6
DNA_SEQUENCE = "ACGTAGCTAGCTAGCTACGATCGATCGATCGATACGATCGATCG"
kmer_vocab = build_kmer_vocabulary(K)

# a. Tokenize the sequence
original_ids = seq_to_kmer_ids(DNA_SEQUENCE, K, kmer_vocab)
print(f"Original sequence has {len(original_ids)} {K}-mer tokens.")

# b. Apply masking
masked_inputs, labels = mask_tokens(original_ids, kmer_vocab)

print("\nOriginal Token IDs (first 15):\n", torch.tensor(original_ids)[:15])
print("\nMasked Input IDs (first 15):\n", masked_inputs[:15])
print("\nLabels (first 15, -100 means not masked):\n", labels[:15])

# The `masked_inputs` tensor is fed to the Transformer.
# The model's output logits are compared against the `labels` tensor
# using CrossEntropyLoss, which conveniently ignores the -100 entries.


## Fine-Tuning: Classification Head and LoRA

**What this step does:** Adapts a pre-trained foundation model to a downstream task with a new head and optional LoRA adapters.

**Input / Output shapes:** Input: token IDs `(batch, seq_len)`. Output: classification logits `(batch, num_labels)`.

> **⚙️ Compute note**
> | Step | Typical wall-clock (CPU / single GPU) | Memory footprint | Bottleneck |
> |------|---------------------------------------|------------------|------------|
> | LoRA fine-tune step (batch 8, len 512) | ~500 ms / ~20 ms | ~2 GB | GPU compute |

> **Optional:** requires extra packages (`transformers`, `peft`, or CUDA for AMP/DDP).


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from transformers import AutoModel # A popular library for pre-trained models
# You may need to install: pip install transformers peft
from peft import get_peft_model, LoraConfig, TaskType

# --- 1. Load a pre-trained model and add a classification head ---
# In a real scenario, this would be a model like 'armheb/dna_bert_6'
# For demonstration, we'll use a generic BERT model.
try:
    base_model = AutoModel.from_pretrained("bert-base-uncased")
except OSError: # Handle case where user is offline
    from transformers import BertConfig, BertModel
    base_model = BertModel(BertConfig())


class FineTunedGenomicModel(nn.Module):
    def __init__(self, base_model, num_labels):
        super().__init__()
        self.base = base_model
        # New head for a binary classification task
        self.classifier = nn.Linear(base_model.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask=None):
        outputs = self.base(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0]
        logits = self.classifier(cls_output)
        return logits

model = FineTunedGenomicModel(base_model, num_labels=2)

# --- 2. Configure Optimizer with Differential Learning Rates ---
optimizer_grouped_parameters = [
    {"params": model.base.parameters(), "lr": 1e-5}, # Smaller LR for base
    {"params": model.classifier.parameters(), "lr": 1e-4}, # Larger LR for head
]
optimizer = optim.AdamW(optimizer_grouped_parameters)
print("Optimizer configured with differential learning rates.")

# --- 3. Configure LoRA for Parameter-Efficient Fine-Tuning ---
# Define the LoRA configuration
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS, 
    r=8,  # The rank of the update matrices (a small number)
    lora_alpha=32, # A scaling factor for the LoRA weights
    lora_dropout=0.1,
    target_modules=["query", "key"] # Target the query and key matrices in attention layers
)

# Create a PEFT model with LoRA
peft_model = get_peft_model(model, lora_config)

print("\nTrainable parameters after applying LoRA:")
peft_model.print_trainable_parameters()

# You can now train 'peft_model' as you would a regular PyTorch model.
# Only the LoRA adapter parameters will be updated.


## Optimization: AdamW Training Step

**What this step does:** Optimizers update model weights to minimise prediction error on labeled genomic data.

**Input / Output shapes:** Input: model, batch, labels. Output: scalar loss after one `optimizer.step()`.

> **⚙️ Compute note**
> | Step | Typical wall-clock (CPU / single GPU) | Memory footprint | Bottleneck |
> |------|---------------------------------------|------------------|------------|
> | AdamW step (64×10 linear) | ~1 ms / ~0.1 ms | ~1 MB | Negligible |


In [ ]:
import torch
import torch.optim as optim
import torch.nn as nn

# --- 1. Setup ---
# Assume we have a model, some input data, and some true labels
model = nn.Linear(10, 1) # A simple example model
input_data = torch.randn(64, 10)
true_labels = torch.randn(64, 1)
loss_function = nn.MSELoss() # Mean Squared Error loss

# --- 2. Create the Optimizer ---
# We pass the model's parameters to the optimizer.
# 'lr' is the learning rate.
# 'weight_decay' is a regularization term (more on this next).
# AdamW is a great default choice.
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)

# --- 3. A Single Training Step ---
# a. Clear old gradients
optimizer.zero_grad()

# b. Forward pass: get model predictions
predictions = model(input_data)

# c. Compute the loss
loss = loss_function(predictions, true_labels)

# d. Backward pass: compute gradients
loss.backward()

# e. Update weights: take a step based on the gradients
optimizer.step()

# --- 4. Observe ---
print("Loss after one step:", loss.item())


## Regularization: Dropout and Weight Decay

**What this step does:** Dropout and weight decay reduce overfitting when training data are limited.

**Input / Output shapes:** Input: activation tensor. Output: dropped activations; optimizer with `weight_decay`.

> **⚙️ Compute note**
> | Step | Typical wall-clock (CPU / single GPU) | Memory footprint | Bottleneck |
> |------|---------------------------------------|------------------|------------|
> | Dropout forward (batch 20) | ~0.1 ms | Negligible | Negligible |


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# --- 1. Dropout Example ---
# Dropout is typically applied after activation functions or between linear layers.
# p=0.5 means each neuron has a 50% chance of being zeroed out during training.
# Note: Dropout is automatically disabled during evaluation (model.eval()).
dropout_layer = nn.Dropout(p=0.5)
activations = torch.randn(20, 16) # Example activations from a previous layer
print("Original activations (first 5):\n", activations[0, :5])
dropped_out_activations = dropout_layer(activations)
print("Activations after dropout (first 5):\n", dropped_out_activations[0, :5])


# --- 2. Weight Decay Example ---
# Weight decay is specified when you create the optimizer.
model = nn.Linear(10, 1)

# AdamW correctly decouples weight decay from the gradient update, making it a preferred choice.
optimizer_with_wd = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)

print("\nOptimizer with weight decay:", optimizer_with_wd)


## Batching and Padding Variable-Length Sequences

**What this step does:** Real genomic batches mix sequence lengths; padding aligns them for efficient GPU training.

**Input / Output shapes:** Input: list of variable-length tensors. Output: padded batch `(batch, max_len)`.

> **⚙️ Compute note**
> | Step | Typical wall-clock (CPU / single GPU) | Memory footprint | Bottleneck |
> |------|---------------------------------------|------------------|------------|
> | pad_sequence collate (batch 32) | ~1 ms | ~8 MB | CPU memory copy |


In [ ]:
import torch
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset

# --- 1. Create a dummy dataset with variable length sequences ---
class GenomicDataset(Dataset):
    def __init__(self):
        self.data = [
            torch.tensor([0, 1, 2, 3], dtype=torch.long),       # len 4
            torch.tensor([0, 1, 2], dtype=torch.long),         # len 3
            torch.tensor([0, 1, 2, 3, 2, 1], dtype=torch.long),  # len 6
        ]
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        return self.data[idx]

# --- 2. Define the custom collate function ---
# This function will be called by the DataLoader for each batch.
def pad_collate_fn(batch):
    # 'batch' is a list of tensors (our sequences)
    # We define our padding token's index (e.g., 4 for 'N')
    padding_value = 4
    
    # pad_sequence stacks the tensors and pads them to the longest sequence in the batch
    padded_batch = pad_sequence(batch, batch_first=True, padding_value=padding_value)
    return padded_batch

# --- 3. Use it with a DataLoader ---
dataset = GenomicDataset()
# batch_size=3 will grab all our data in one go
data_loader = DataLoader(dataset, batch_size=3, collate_fn=pad_collate_fn)

# --- 4. Observe the Output ---
# Get one batch from the loader
padded_sequences = next(iter(data_loader))

print("Padded batch of sequences:\n", padded_sequences)
print("Shape of the batch:", padded_sequences.shape)


## Loss Functions for Genomic Classification

**What this step does:** Binary and multi-class losses score how well predictions match binding or structure labels.

**Input / Output shapes:** Input: logits and labels. Output: scalar loss.

> **⚙️ Compute note**
> | Step | Typical wall-clock (CPU / single GPU) | Memory footprint | Bottleneck |
> |------|---------------------------------------|------------------|------------|
> | BCE / CE loss (batch 4) | ~0.05 ms | Negligible | Negligible |


In [ ]:
import torch
import torch.nn as nn

# --- 1. Binary Classification Example ---
# Task: Predict if 4 sequences are binding sites (1) or not (0).
# Model outputs raw scores (logits). Positive scores -> class 1, Negative -> class 0.
logits = torch.tensor([-2.5, 4.1, -0.5, 1.1]) # Raw output from a model for a batch of 4
true_labels = torch.tensor([0.0, 1.0, 0.0, 1.0]) # The ground truth (as floats)

# BCEWithLogitsLoss is best for binary tasks.
loss_fn_bce = nn.BCEWithLogitsLoss()
binary_loss = loss_fn_bce(logits, true_labels)
print(f"Binary Cross-Entropy Loss: {binary_loss.item():.4f}")


# --- 2. Multi-Class Classification Example ---
# Task: Classify 2 residues into one of 3 classes (helix, sheet, coil).
# Model outputs a score for each class.
logits_multi = torch.tensor([
    [3.2, -1.0, 0.5],  # Logits for residue 1 (class 0 is highest)
    [-0.8, 2.5, 0.1]   # Logits for residue 2 (class 1 is highest)
])
true_labels_multi = torch.tensor([0, 1], dtype=torch.long)  # The ground truth (as integers)

# CrossEntropyLoss is best for multi-class tasks.
loss_fn_ce = nn.CrossEntropyLoss()
multiclass_loss = loss_fn_ce(logits_multi, true_labels_multi)
print(f"Multi-Class Cross-Entropy Loss: {multiclass_loss.item():.4f}")


## Mixed Precision Training

**What this step does:** FP16 autocast speeds GPU training and reduces memory without changing the learning objective.

**Input / Output shapes:** Input: model batch on CUDA. Output: loss after scaled backward pass.

> **⚙️ Compute note**
> | Step | Typical wall-clock (CPU / single GPU) | Memory footprint | Bottleneck |
> |------|---------------------------------------|------------------|------------|
> | AMP training step (batch 64) | ~2 ms / ~0.5 ms | ~half vs FP32 | GPU tensor cores |

> **Optional:** requires extra packages (`transformers`, `peft`, or CUDA for AMP/DDP).


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.cuda.amp import autocast, GradScaler

# This code requires a CUDA-enabled GPU.

# --- 1. Setup ---
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cpu":
    print("Mixed precision requires a CUDA GPU. Skipping example.")
else:
    model = nn.Linear(10, 1).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=0.001)
    loss_fn = nn.MSELoss()
    
    # Create a GradScaler once at the beginning of training.
    scaler = GradScaler()

    # Dummy data
    input_data = torch.randn(64, 10, device=device)
    true_labels = torch.randn(64, 1, device=device)

    # --- 2. A Single Mixed-Precision Training Step ---
    optimizer.zero_grad()

    # Wrap the forward pass with autocast.
    # Operations inside this block will run in lower precision where possible.
    with autocast(device_type='cuda', dtype=torch.float16):
        predictions = model(input_data)
        loss = loss_fn(predictions, true_labels)

    # Scale the loss and call backward() on the scaled loss.
    scaler.scale(loss).backward()

    # scaler.step() first unscales the gradients and then calls optimizer.step().
    scaler.step(optimizer)

    # Update the scale for the next iteration.
    scaler.update()

    print("Training step completed with mixed precision.")
    print("Loss:", loss.item())


## Distributed Data Parallel Training

**What this step does:** DDP shards data across GPUs so large genomic corpora train in parallel.

**Input / Output shapes:** Input: per-rank data shard. Output: synchronised gradients across processes.

> **⚙️ Compute note**
> | Step | Typical wall-clock (CPU / single GPU) | Memory footprint | Bottleneck |
> |------|---------------------------------------|------------------|------------|
> | DDP all-reduce (4 GPUs) | ~5 ms / ~1 ms | 4× model replicas | Network sync |

> **Optional:** requires extra packages (`transformers`, `peft`, or CUDA for AMP/DDP).


In [ ]:
# This is a conceptual script, e.g., `my_ddp_script.py`
# You would run it from the command line like:
# torchrun --nproc_per_node=4 my_ddp_script.py

import os
import torch
import torch.nn as nn
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data.distributed import DistributedSampler
from torch.utils.data import DataLoader, Dataset

def setup(rank, world_size):
    """Initializes the distributed process group."""
    os.environ['MASTER_ADDR'] = 'localhost'
    os.environ['MASTER_PORT'] = '12355'
    dist.init_process_group("nccl", rank=rank, world_size=world_size)

def cleanup():
    """Cleans up the distributed process group."""
    dist.destroy_process_group()

# Dummy class for the example to be runnable
class MyDataset(Dataset):
    def __len__(self): return 1000
    def __getitem__(self, idx): return torch.randn(10), torch.randn(1)

def main_worker(rank, world_size):
    """The main training function for each process."""
    setup(rank, world_size)
    
    # 1. Wrap the model with DDP
    # The model is moved to the GPU corresponding to the process's rank.
    model = nn.Linear(10, 1).to(rank)
    ddp_model = DDP(model, device_ids=[rank])

    # 2. Use DistributedSampler for the DataLoader
    # This ensures each process gets a different slice of the data.
    dataset = MyDataset() 
    sampler = DistributedSampler(dataset, num_replicas=world_size, rank=rank)
    # Note: shuffle=False because the sampler handles shuffling.
    loader = DataLoader(dataset, batch_size=32, sampler=sampler, shuffle=False) 

    # ... standard training loop using `ddp_model` ...
    # The gradient synchronization is handled automatically by DDP during loss.backward().

    # Only save the model on the main process (rank 0) to avoid conflicts.
    if rank == 0:
        torch.save(ddp_model.state_dict(), "my_model.pt")

    cleanup()

if __name__ == '__main__':
    # This part is handled by torchrun, which sets environment variables.
    # world_size = number of GPUs
    # rank = the ID of the current GPU (0, 1, 2, ...)
    # For a conceptual example, we'll skip the process spawning part.
    print("To run a DDP script, use 'torchrun'.")
    print("This example shows the key code modifications needed within the script.")


## End-to-End Genomic Classifier

**What this step does:** Combines embedding, convolution, pooling, and a linear head to detect a simple ACG motif.

**Input / Output shapes:** Input: `(batch, seq_len)` token IDs. Output: binding probability `(batch, 1)`.

> **⚙️ Compute note**
> | Step | Typical wall-clock (CPU / single GPU) | Memory footprint | Bottleneck |
> |------|---------------------------------------|------------------|------------|
> | E2E training epoch (2k seqs, CPU) | ~30 s / ~3 s | ~200 MB | Conv + linear |


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np

# --- 1. Model Definition ---
class SimpleGenomicClassifier(nn.Module):
    def __init__(self, num_tokens=5, embedding_dim=32, seq_len=101):
        super().__init__()
        self.embedding = nn.Embedding(num_embeddings=num_tokens, embedding_dim=embedding_dim)
        self.conv1d = nn.Conv1d(in_channels=embedding_dim, out_channels=64, kernel_size=8)
        self.relu = nn.ReLU()
        self.flatten = nn.Flatten()

        # Calculate the size of the flattened output after convolution and pooling
        with torch.no_grad():
            dummy_input = torch.zeros(1, seq_len, dtype=torch.long)
            dummy_embedded = self.embedding(dummy_input).permute(0, 2, 1)
            dummy_conv = self.conv1d(dummy_embedded)
            pool_kernel_size = dummy_conv.shape[2]  # Global Max Pooling
            dummy_pool = nn.MaxPool1d(kernel_size=pool_kernel_size)(dummy_conv)
            dummy_flattened = self.flatten(dummy_pool)
            flattened_size = dummy_flattened.shape[1]

        self.linear = nn.Linear(flattened_size, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.embedding(x) # (B, L) -> (B, L, D)
        x = x.permute(0, 2, 1) # (B, L, D) -> (B, D, L) for Conv1d
        x = self.conv1d(x) # (B, D, L) -> (B, C_out, L_out)
        x = self.relu(x)
        
        # Global Max Pooling
        pool_kernel_size = x.shape[2]
        x = nn.MaxPool1d(kernel_size=pool_kernel_size)(x)
        
        x = self.flatten(x) # (B, C_out, 1) -> (B, C_out)
        x = self.linear(x) # (B, C_out) -> (B, 1)
        x = self.sigmoid(x)
        return x

# --- 2. Data Preparation ---
def generate_synthetic_data(num_samples=1000, seq_len=101):
    # Vocab: 0=PAD, 1=A, 2=C, 3=G, 4=T
    sequences = np.random.randint(1, 5, size=(num_samples, seq_len))
    # Labels: 1 if '123' (ACG) is in the sequence, 0 otherwise
    labels = np.array([1 if any(np.array_equal(sequences[i, j:j+3], [1, 2, 3]) for j in range(seq_len - 2)) else 0 for i in range(num_samples)])
    return torch.tensor(sequences, dtype=torch.long), torch.tensor(labels, dtype=torch.float32).unsqueeze(1)

# --- 3. Training Loop ---
if __name__ == '__main__':
    # Hyperparameters
    SEQ_LEN = 101
    NUM_SAMPLES = 2000
    BATCH_SIZE = 64
    EPOCHS = 5
    LEARNING_RATE = 0.001

    # Generate data and create DataLoader
    X, y = generate_synthetic_data(num_samples=NUM_SAMPLES, seq_len=SEQ_LEN)
    dataset = TensorDataset(X, y)
    train_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

    # Instantiate model, loss function, and optimizer
    model = SimpleGenomicClassifier(seq_len=SEQ_LEN)
    criterion = nn.BCELoss() # For binary classification with sigmoid output
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

    print("Starting training...")
    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0
        for sequences, labels in train_loader:
            optimizer.zero_grad()
            outputs = model(sequences)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        
        avg_loss = total_loss / len(train_loader)
        print(f"Epoch [{epoch+1}/{EPOCHS}], Loss: {avg_loss:.4f}")

    print("Training finished.")


## Evaluation Metrics for Imbalanced Genomic Data

**What this step does:** ROC and precision-recall curves assess rare-class performance beyond misleading accuracy.

**Input / Output shapes:** Input: `y_true`, `y_scores`. Output: ROC/PR plots and AUC metrics.

> **⚙️ Compute note**
> | Step | Typical wall-clock (CPU / single GPU) | Memory footprint | Bottleneck |
> |------|---------------------------------------|------------------|------------|
> | ROC + PR plot (20 samples) | ~300 ms | ~5 MB | CPU plotting |


In [ ]:
import numpy as np
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score
import matplotlib.pyplot as plt

# Example ground truth labels and model prediction scores
# An imbalanced dataset with 5 positive samples out of 20 (25%)
y_true = np.array([0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0])
y_scores = np.array([0.1, 0.4, 0.35, 0.8, 0.2, 0.3, 0.4, 0.1, 0.05, 0.15, 0.7, 0.5, 0.6, 0.2, 0.3, 0.4, 0.9, 0.3, 0.75, 0.25])

# --- ROC Curve ---
fpr, tpr, _ = roc_curve(y_true, y_scores)
roc_auc = auc(fpr, tpr)

# --- Precision-Recall Curve ---
precision, recall, _ = precision_recall_curve(y_true, y_scores)
pr_auc = average_precision_score(y_true, y_scores)

# --- Plotting ---
plt.figure(figsize=(12, 5))

# Plot ROC Curve
plt.subplot(1, 2, 1)
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc="lower right")

# Plot Precision-Recall Curve
plt.subplot(1, 2, 2)
# Plot the no-skill line (prevalence of the positive class)
no_skill = len(y_true[y_true==1]) / len(y_true)
plt.plot([0, 1], [no_skill, no_skill], linestyle='--', color='navy', label='No-Skill')
plt.plot(recall, precision, color='blue', lw=2, label=f'PR curve (AUC = {pr_auc:.2f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend(loc="lower left")

plt.tight_layout()
plt.show()

print(f"AUC-ROC: {roc_auc:.4f}")
print(f"PR-AUC (Average Precision): {pr_auc:.4f}")
